# 04 · Derivatives, excess enthalpy and response functions

**Requirements:** SAFT/JAX and RDKit.

Run this notebook from top to bottom in the environment where ThermoWeave is installed. All examples are self-contained unless explicitly marked optional. See the [course index](../tutorials.md) and [theory guide](../theory.md).

<a class="md-button" href="04_derivatives_and_response_functions.ipynb" download>Download notebook</a>


The model's composition Jacobian is $J_{ij}=\partial\ln\gamma_i/\partial n_j$
 at fixed T, P and other mole numbers, evaluated at n=x. It is **not** a matrix
 of independent mole-fraction derivatives. Homogeneity and Gibbs–Duhem imply
 $Jx=0$, $x^TJ=0$ and $J=J^T$. Density roots are differentiated implicitly.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from thermoweave import SAFTgammaMie
from thermoweave.saft import kernels
from thermoweave.saft.params import R
model = SAFTgammaMie.from_smiles({'water': 'O', 'ethanol': 'CCO'},
                               pressure_Pa=101325., report_assignment=False)
T, P, x = 298.15, 101325., np.array([0.8, 0.2])
J = model.ln_activity_coefficients_jacobian(T, x)
print('d ln gamma / d n:', J)
np.testing.assert_allclose(J @ x, 0, atol=1e-10)
np.testing.assert_allclose(x @ J, 0, atol=1e-10)
np.testing.assert_allclose(J, J.T, atol=1e-10)

d ln gamma / d n: [[-0.12806386  0.51225543]
 [ 0.51225543 -2.04902171]]


## Excess enthalpy by a temperature derivative
 At fixed P and x, $h^E=-RT^2\sum_i x_i(\partial\ln\gamma_i/\partial T)$.
 The pure-liquid reference contributions are already included in ln γ.
 Direct kernels require float64; public model methods enable it locally.

In [2]:
with jax.enable_x64(True):
    p = model.kernel_params
    derivative = jax.jit(jax.jacfwd(kernels.ln_activity_coefficients, argnums=1))
    dlng_dT = np.asarray(derivative(p, jnp.float64(T), jnp.asarray(x), jnp.float64(P)))
h_excess = -R*T*T*float(x @ dlng_dT)
print('d ln gamma / d T [1/K]:', dlng_dT)
print('Excess enthalpy [J/mol]:', h_excess)
assert np.isfinite(h_excess)

d ln gamma / d T [1/K]: [7.95891572e-05 1.59204961e-03]
Excess enthalpy [J/mol]: -282.3966642746281


## Partial volumes, compressibility and Kirkwood–Buff integrals
 $\bar v_i=\partial V/\partial n_i$, $v=\sum_i x_i\bar v_i$, and
 $\kappa_T=[\rho(\partial P/\partial\rho)_{T,x}]^{-1}$.
 The KB helper returns **cm³/mol** for volumes and **1/GPa** for compressibility.
 Its thermodynamic inversion describes homogeneous concentration fluctuations;
 it is not a molecular simulation and should be interpreted in a stable liquid.

In [3]:
kb = model.kirkwood_buff_integrals(T, x)
print('Partial molar volumes [cm3/mol]:', kb['partial_volumes'])
print('Molar volume [cm3/mol]:', kb['V'])
print('Isothermal compressibility [1/GPa]:', kb['kappa_T'])
print('KB integrals [cm3/mol]:', kb['G'])
np.testing.assert_allclose(x @ kb['partial_volumes'], 1e6/model.density(T, x), rtol=1e-9)
np.testing.assert_allclose(kb['G'], kb['G'].T, atol=1e-8)
assert kb['kappa_T'] > 0

Partial molar volumes [cm3/mol]: [17.74382006 54.8915433 ]
Molar volume [cm3/mol]: 25.17336471071279
Isothermal compressibility [1/GPa]: 0.3332346973305944
KB integrals [cm3/mol]: [[ 30.70973733 -78.50052761]
 [-78.50052761 -22.47070861]]


**Try:** vary composition away from the pure-component endpoints. Values
 near a spinodal can be very large. Numerical differentiation and thermodynamic
 identities check consistency; experimental agreement requires validation data.
 The same JAX kernels can differentiate pressure or residual Helmholtz energy
 with respect to temperature, density and fitted parameters.